# AI Personal Tutor

Upload a lecture PDF, ask questions, and create a quiz from it. Run the cells in order.

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
!pip install -q "transformers==4.52.4" "bitsandbytes>=0.46.1" accelerate langchain langchain-classic langchain-community langchain-core langchain-huggingface langchain-text-splitters pypdf faiss-cpu sentence-transformers fastapi uvicorn pyngrok requests python-multipart


In [ ]:
import os
import re
import hmac
import tempfile
import threading
import time
from pathlib import Path

import torch
import uvicorn
from fastapi import Depends, FastAPI, File, HTTPException, UploadFile
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from kaggle_secrets import UserSecretsClient
from pydantic import BaseModel, Field


In [ ]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from pyngrok import ngrok


In [ ]:
model_name = "mistralai/Mistral-Nemo-Instruct-2407"

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)


In [ ]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("Model and pipeline loaded")


In [ ]:
pdf_files = list(Path("/kaggle/input").rglob("*.pdf"))

if len(pdf_files) == 0:
    raise FileNotFoundError("Attach a PDF dataset to this notebook first.")

pdf_path = str(pdf_files[0])
print("PDF:", pdf_path)


In [ ]:
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print("Pages loaded:", len(documents))


In [ ]:
def remove_empty_and_duplicate_pages(page_documents):
    cleaned_documents = []
    seen_pages = set()

    for document in page_documents:
        page_text = " ".join(document.page_content.split())
        if len(page_text) < 20 or page_text in seen_pages:
            continue
        seen_pages.add(page_text)
        cleaned_documents.append(document)

    return cleaned_documents

documents = remove_empty_and_duplicate_pages(documents)
print("Readable pages:", len(documents))


In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
chunks = text_splitter.split_documents(documents)
print("Chunks:", len(chunks))


In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectordb = FAISS.from_documents(chunks, embedding)
print("FAISS index created")


In [ ]:
def retrieve_context(query, k=4):
    return vectordb.similarity_search(query, k=k)


In [ ]:
api_token = UserSecretsClient().get_secret("CV_API_TOKEN")
app = FastAPI()
bearer_scheme = HTTPBearer(auto_error=False)

def check_api_token(credentials):
    if credentials is None or not hmac.compare_digest(
        credentials.credentials, api_token
    ):
        raise HTTPException(status_code=401, detail="Unauthorized")


In [ ]:
class ChatRequest(BaseModel):
    topic: str
    history: list[dict] = Field(default_factory=list)


In [ ]:
explanation_prompt = PromptTemplate(
    input_variables=["context", "history", "topic"],
    template="""
You are a tutor helping a student understand lecture notes.
Use only the lecture context for facts. Do not add outside information.
Use the conversation history to understand follow-up questions.
If the answer is not in the context, say so.
Answer in the same language as the student's question.

Conversation history:
{history}

Lecture context:
{context}

Student question: {topic}
Answer:
""",
)


In [ ]:
def explain_topic(topic, history):
    search_text = topic
    for turn in history[-6:]:
        if turn.get("role") == "user":
            search_text += " " + turn.get("content", "")

    source_documents = retrieve_context(search_text)
    if not source_documents:
        return "I could not find relevant information in the lecture."

    context = ""
    pages = set()
    for document in source_documents:
        page = document.metadata.get("page", 0) + 1
        pages.add(page)
        context += f"[Page {page}]\n{document.page_content}\n\n"

    history_text = "\n".join(
        f"{item.get('role')}: {item.get('content')}"
        for item in history[-6:]
    )
    prompt = explanation_prompt.format(
        context=context, history=history_text, topic=topic
    )
    result = generator(
        prompt, max_new_tokens=300, do_sample=False,
        return_full_text=False, pad_token_id=tokenizer.eos_token_id
    )
    answer = result[0]["generated_text"].strip()
    page_text = ", ".join(str(page) for page in sorted(pages))
    return f"{answer}\n\nSource pages: {page_text}"


In [ ]:
@app.post("/chat")
def chat_endpoint(
    request: ChatRequest,
    credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme),
):
    check_api_token(credentials)
    return {"explanation": explain_topic(request.topic, request.history)}


In [ ]:
quiz_schema = ResponseSchema(
    name="questions",
    description=(
        "A list of questions. Each question has question, options (four choices), "
        "correct_answer (A, B, C, or D), explanation, and source_pages."
    ),
)
quiz_parser = StructuredOutputParser.from_response_schemas([quiz_schema])
format_instructions = quiz_parser.get_format_instructions(only_json=True)


In [ ]:
class QuizRequest(BaseModel):
    scope: str = "all"
    num_questions: int = Field(gt=0)
    topic: str = ""


In [ ]:
quiz_prompt = PromptTemplate(
    input_variables=["context", "topic", "num_questions", "format_instructions"],
    template="""
Create {num_questions} multiple-choice questions using only the lecture context.
If a topic is given, focus on it: {topic}
Each question needs four options, A to D, and exactly one correct answer.
Use plausible but incorrect distractors. Do not add outside facts.
Include a short explanation and the source page number.
Return JSON only, following this format:
{format_instructions}

Lecture context:
{context}
""",
)


In [ ]:
def extract_json(text):
    match = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        return text[start:end + 1]
    return text.strip()


In [ ]:
def generate_quiz(num_questions, topic=""):
    context = ""
    available_pages = set()
    for chunk in chunks:
        page = chunk.metadata.get("page", 0) + 1
        available_pages.add(page)
        context += f"[Page {page}]\n{chunk.page_content}\n\n"

    for attempt in range(2):
        prompt = quiz_prompt.format(
            context=context, topic=topic or "the full lecture",
            num_questions=num_questions,
            format_instructions=format_instructions,
        )
        if attempt == 1:
            prompt += "\nReturn one valid JSON object only."

        try:
            response = generator(
                prompt, max_new_tokens=300 + num_questions * 120,
                do_sample=False, return_full_text=False,
                pad_token_id=tokenizer.eos_token_id,
            )[0]["generated_text"]
            quiz = quiz_parser.parse(extract_json(response))
            questions = quiz.get("questions", [])
            if len(questions) != num_questions:
                raise ValueError("The model returned the wrong number of questions.")

            for question in questions:
                if len(question.get("options", [])) != 4:
                    raise ValueError("Each question must have four options.")
                if question.get("correct_answer", "").upper() not in ["A", "B", "C", "D"]:
                    raise ValueError("The correct answer must be A, B, C, or D.")
                page_values = question.get("source_pages", [])
                if not isinstance(page_values, list):
                    page_values = [page_values]
                question_pages = []
                for value in page_values:
                    match = re.search(r"\d+", str(value))
                    if match and int(match.group()) in available_pages:
                        question_pages.append(int(match.group()))
                if not question_pages:
                    raise ValueError("A question is missing a source page.")
                question["source_pages"] = sorted(set(question_pages))

            return quiz
        except Exception as error:
            if attempt == 1:
                return {"error": f"Quiz could not be created: {error}"}


In [ ]:
@app.post("/quiz_by_scope")
def quiz_endpoint(
    request: QuizRequest,
    credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme),
):
    check_api_token(credentials)
    if request.scope != "all":
        raise HTTPException(status_code=400, detail="Choose the entire PDF.")
    return generate_quiz(request.num_questions, request.topic)


In [ ]:
@app.post("/upload")
async def upload_pdf(
    file: UploadFile = File(...),
    credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme),
):
    check_api_token(credentials)
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Upload a PDF file.")

    pdf_bytes = await file.read()
    if not pdf_bytes:
        raise HTTPException(status_code=400, detail="The PDF is empty.")

    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_file:
            temp_file.write(pdf_bytes)
            temp_path = temp_file.name
        new_documents = PyPDFLoader(temp_path).load()
    except Exception as error:
        raise HTTPException(status_code=400, detail=f"Could not read PDF: {error}")
    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

    pages_loaded = len(new_documents)
    new_documents = remove_empty_and_duplicate_pages(new_documents)
    new_chunks = text_splitter.split_documents(new_documents)
    if not new_chunks:
        raise HTTPException(status_code=400, detail="No readable text in PDF.")
    new_vectordb = FAISS.from_documents(new_chunks, embedding)

    indexed_pages = sorted({chunk.metadata.get("page", 0) + 1 for chunk in new_chunks})
    global documents, chunks, vectordb
    documents, chunks, vectordb = new_documents, new_chunks, new_vectordb
    return {
        "message": "PDF uploaded and indexed successfully.",
        "pages_loaded": pages_loaded,
        "indexed_pages": indexed_pages,
        "chunks_created": len(new_chunks),
    }


In [ ]:
study_guide_schema = [
    ResponseSchema(
        name="summary",
        description="A short summary of the selected lecture content.",
    ),
    ResponseSchema(
        name="key_points",
        description="A list of four to six important points from the lecture.",
    ),
    ResponseSchema(
        name="source_pages",
        description="A list of lecture page numbers used in the summary.",
    ),
]

study_guide_parser = StructuredOutputParser.from_response_schemas(
    study_guide_schema
)
study_guide_format = study_guide_parser.get_format_instructions(only_json=True)

print("Study guide schema is ready.")


In [ ]:
study_guide_prompt = PromptTemplate(
    input_variables=["context", "focus", "format_instructions"],
    template="""
You are a patient tutor making a study guide from lecture notes.

Use only facts stated in the lecture context.
Write a concise summary and four to six key points.
If a focus topic is provided, focus the summary and points on it.
Do not add outside facts.
Include only page numbers that appear in the lecture context.
Return exactly one valid JSON object using the required format.

Lecture context:
{context}

Focus:
{focus}

Required JSON format:
{format_instructions}
""",
)

print("Study guide prompt is ready.")


In [ ]:
class StudyGuideRequest(BaseModel):
    focus_topic: str = ""


class FlashcardsRequest(BaseModel):
    num_cards: int = Field(gt=0, le=20)
    topic: str = ""


In [ ]:
def full_lecture_context():
    context_parts = []
    available_pages = set()

    for chunk in chunks:
        page_number = chunk.metadata.get("page", 0) + 1
        available_pages.add(page_number)
        context_parts.append(
            f"[Page {page_number}]\n{chunk.page_content}"
        )

    return "\n\n".join(context_parts), available_pages


def keep_valid_pages(page_values, available_pages):
    if not isinstance(page_values, list):
        page_values = [page_values]

    valid_pages = []
    for value in page_values:
        match = re.search(r"\d+", str(value))
        if match:
            page_number = int(match.group())
            if page_number in available_pages:
                valid_pages.append(page_number)

    return sorted(set(valid_pages))


print("Lecture context helpers are ready.")


In [ ]:
def create_study_guide(focus_topic=""):
    context, available_pages = full_lecture_context()

    if not context:
        return {"error": "No readable lecture text is available."}

    focus = focus_topic.strip() or "Cover the whole lecture."
    last_error = None

    for attempt in range(2):
        prompt = study_guide_prompt.format(
            context=context,
            focus=focus,
            format_instructions=study_guide_format,
        )
        if attempt == 1:
            prompt += "\nReturn one valid JSON object only."

        try:
            generated = generator(
                prompt,
                max_new_tokens=700,
                do_sample=False,
                return_full_text=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            result = study_guide_parser.parse(
                extract_json(generated[0]["generated_text"])
            )

            key_points = result.get("key_points")
            if not isinstance(key_points, list) or not key_points:
                raise ValueError("The response has no key points.")

            source_pages = keep_valid_pages(
                result.get("source_pages", []),
                available_pages,
            )
            if not source_pages:
                raise ValueError("The response has no valid source pages.")

            return {
                "summary": str(result.get("summary", "")).strip(),
                "key_points": [str(point).strip() for point in key_points],
                "source_pages": source_pages,
            }

        except Exception as error:
            last_error = error

    return {
        "error": (
            "The study guide could not be created after one retry. "
            f"{type(last_error).__name__}: {last_error}"
        )
    }


In [ ]:
@app.post("/study_guide")
def study_guide_endpoint(
    request: StudyGuideRequest,
    credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme),
):
    check_api_token(credentials)
    return create_study_guide(request.focus_topic)


print("Study guide endpoint is ready.")


In [ ]:
flashcards_schema = [
    ResponseSchema(
        name="cards",
        description=(
            "A list of flashcards. Each card has question, answer, "
            "and source_pages fields."
        ),
    ),
]

flashcards_parser = StructuredOutputParser.from_response_schemas(
    flashcards_schema
)
flashcards_format = flashcards_parser.get_format_instructions(only_json=True)

flashcards_prompt = PromptTemplate(
    input_variables=[
        "context",
        "topic",
        "num_cards",
        "format_instructions",
    ],
    template="""
You are a patient tutor creating study flashcards from lecture notes.

Create exactly {num_cards} question-and-answer flashcards using only the lecture.
If a focus topic is provided, focus on it: {topic}
Each card must test one useful idea.
Keep each answer short and accurate.
For each card, include source_pages as a list of page numbers from the context.
Do not use outside facts.
Return exactly one valid JSON object in the required format.

Lecture context:
{context}

Required JSON format:
{format_instructions}
""",
)

print("Flashcards schema and prompt are ready.")


In [ ]:
def create_flashcards(num_cards, topic=""):
    context, available_pages = full_lecture_context()

    if not context:
        return {"error": "No readable lecture text is available."}

    last_error = None

    for attempt in range(2):
        prompt = flashcards_prompt.format(
            context=context,
            topic=topic.strip() or "Cover the lecture broadly.",
            num_cards=num_cards,
            format_instructions=flashcards_format,
        )
        if attempt == 1:
            prompt += "\nReturn one valid JSON object only."

        try:
            generated = generator(
                prompt,
                max_new_tokens=350 + num_cards * 100,
                do_sample=False,
                return_full_text=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            result = flashcards_parser.parse(
                extract_json(generated[0]["generated_text"])
            )

            cards = result.get("cards")
            if not isinstance(cards, list) or len(cards) != num_cards:
                received = len(cards) if isinstance(cards, list) else 0
                raise ValueError(
                    f"Expected {num_cards} flashcards, received {received}."
                )

            clean_cards = []
            for card in cards:
                question = str(card.get("question", "")).strip()
                answer = str(card.get("answer", "")).strip()
                source_pages = keep_valid_pages(
                    card.get("source_pages", []),
                    available_pages,
                )

                if not question or not answer or not source_pages:
                    raise ValueError(
                        "A flashcard is missing a question, answer, or valid page."
                    )

                clean_cards.append(
                    {
                        "question": question,
                        "answer": answer,
                        "source_pages": source_pages,
                    }
                )

            return {"cards": clean_cards}

        except Exception as error:
            last_error = error

    return {
        "error": (
            "Flashcards could not be created after one retry. "
            f"{type(last_error).__name__}: {last_error}"
        )
    }


In [ ]:
@app.post("/flashcards")
def flashcards_endpoint(
    request: FlashcardsRequest,
    credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme),
):
    check_api_token(credentials)
    return create_flashcards(request.num_cards, request.topic)


print("Flashcards endpoint is ready.")


In [ ]:
if "server_thread" not in globals() or not server_thread.is_alive():
    server = uvicorn.Server(
        uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    )
    server_thread = threading.Thread(target=server.run, daemon=True)
    server_thread.start()
    time.sleep(2)
    print("Tutor API server started on port 8000")
else:
    print("Tutor API server is already running")


In [ ]:
ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)
tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url

print("Public API URL:", public_url)
print("Chat:", public_url + "/chat")
print("Study guide:", public_url + "/study_guide")
print("Flashcards:", public_url + "/flashcards")
print("Quiz:", public_url + "/quiz_by_scope")
print("Upload:", public_url + "/upload")
